
# Healthcare Patient Analytics

## Overview
This notebook implements patient and encounter analytics
using PySpark, Spark SQL, and Delta Lake on Databricks.

## Data Sources
Synthetic patient and healthcare encounter datasets.
No real patient information is used.

## Data Processing
- Explicit schema definitions
- Primary key validation
- Data quality checks
- Referential integrity validation
- PySpark joins and aggregations
- Delta Lake managed tables
- SQL-based healthcare analytics

## Data Outputs
- workspace.healthcare_analytics.patients
- workspace.healthcare_analytics.encounters
- workspace.healthcare_analytics.patient_analytics

## Quality Assurance
The notebook validates record counts, uniqueness,
referential integrity, financial reconciliation,
and Delta table storage formats.

## Technology Stack
- Databricks
- Apache Spark
- PySpark
- Spark SQL
- Delta Lake
- Python


In [0]:
%python

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

from datetime import date
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("healthcare_analytics")

print("Databricks healthcare analytics initialized")
print("Spark version:", spark.version)
print("Current catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current schema:", spark.sql("SELECT current_schema()").first()[0])


Databricks healthcare analytics initialized
Spark version: 4.2.0
Current catalog: workspace
Current schema: default


In [0]:
%python

catalog_name = "workspace"
schema_name = "healthcare_analytics"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`"
)

spark.sql(
    f"USE CATALOG `{catalog_name}`"
)

spark.sql(
    f"USE SCHEMA `{schema_name}`"
)

print("Healthcare analytics schema initialized")
print("Active catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Active schema:", spark.sql("SELECT current_schema()").first()[0])

spark.sql("SHOW SCHEMAS").show(truncate=False)


Healthcare analytics schema initialized
Active catalog: workspace
Active schema: healthcare_analytics
+--------------------+
|databaseName        |
+--------------------+
|default             |
|healthcare_analytics|
|information_schema  |
+--------------------+



In [0]:
%python

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType
)

from datetime import date

# Define the synthetic patient data schema
patient_schema = StructType([
    StructField("patient_id", StringType(), False),
    StructField("patient_name", StringType(), False),
    StructField("age", IntegerType(), False),
    StructField("gender", StringType(), False),
    StructField("city", StringType(), False),
    StructField("registration_date", DateType(), False)
])

# Generate synthetic healthcare patient records
patient_data = [
    ("P001", "Patient Alpha", 34, "Female", "Bengaluru", date(2026, 1, 10)),
    ("P002", "Patient Beta", 46, "Male", "Mysuru", date(2026, 1, 12)),
    ("P003", "Patient Gamma", 28, "Female", "Hyderabad", date(2026, 2, 3)),
    ("P004", "Patient Delta", 62, "Male", "Chennai", date(2026, 2, 15)),
    ("P005", "Patient Epsilon", 51, "Female", "Pune", date(2026, 3, 4))
]

# Create the PySpark DataFrame
patients_df = spark.createDataFrame(
    patient_data,
    schema=patient_schema
)

# Display the synthetic patient dataset
display(patients_df)

# Verify record count and schema
print("Total patient records:", patients_df.count())

patients_df.printSchema()


patient_id,patient_name,age,gender,city,registration_date
P001,Patient Alpha,34,Female,Bengaluru,2026-01-10
P002,Patient Beta,46,Male,Mysuru,2026-01-12
P003,Patient Gamma,28,Female,Hyderabad,2026-02-03
P004,Patient Delta,62,Male,Chennai,2026-02-15
P005,Patient Epsilon,51,Female,Pune,2026-03-04


Total patient records: 5
root
 |-- patient_id: string (nullable = false)
 |-- patient_name: string (nullable = false)
 |-- age: integer (nullable = false)
 |-- gender: string (nullable = false)
 |-- city: string (nullable = false)
 |-- registration_date: date (nullable = false)



In [0]:
%python

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    DateType
)

from datetime import date

# Define the healthcare encounter schema
encounter_schema = StructType([
    StructField("encounter_id", StringType(), False),
    StructField("patient_id", StringType(), False),
    StructField("department", StringType(), False),
    StructField("encounter_date", DateType(), False),
    StructField("treatment_cost", DoubleType(), False)
])

# Generate synthetic healthcare encounter records
encounter_data = [
    ("E001", "P001", "Cardiology", date(2026, 4, 1), 2500.0),
    ("E002", "P002", "Neurology", date(2026, 4, 2), 4200.0),
    ("E003", "P001", "General Medicine", date(2026, 4, 5), 1200.0),
    ("E004", "P003", "Orthopedics", date(2026, 4, 6), 3600.0),
    ("E005", "P004", "Cardiology", date(2026, 4, 7), 5400.0),
    ("E006", "P005", "General Medicine", date(2026, 4, 8), 1800.0),
    ("E007", "P002", "Cardiology", date(2026, 4, 9), 3100.0)
]

# Create the encounter DataFrame
encounters_df = spark.createDataFrame(
    encounter_data,
    schema=encounter_schema
)

# Display the encounter dataset
display(encounters_df)

# Verify the number of records and schema
print("Total encounter records:", encounters_df.count())

encounters_df.printSchema()


encounter_id,patient_id,department,encounter_date,treatment_cost
E001,P001,Cardiology,2026-04-01,2500.0
E002,P002,Neurology,2026-04-02,4200.0
E003,P001,General Medicine,2026-04-05,1200.0
E004,P003,Orthopedics,2026-04-06,3600.0
E005,P004,Cardiology,2026-04-07,5400.0
E006,P005,General Medicine,2026-04-08,1800.0
E007,P002,Cardiology,2026-04-09,3100.0


Total encounter records: 7
root
 |-- encounter_id: string (nullable = false)
 |-- patient_id: string (nullable = false)
 |-- department: string (nullable = false)
 |-- encounter_date: date (nullable = false)
 |-- treatment_cost: double (nullable = false)



In [0]:
%python

from pyspark.sql import functions as F
import logging

logger = logging.getLogger("healthcare_analytics")

# Reusable data quality validation function
def validate_dataset(df, primary_key):
    total_records = df.count()

    null_keys = df.filter(
        F.col(primary_key).isNull()
    ).count()

    duplicate_keys = (
        df.groupBy(primary_key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if total_records == 0:
        raise ValueError("Dataset cannot be empty")

    if null_keys > 0:
        raise ValueError(
            f"Null primary keys found: {null_keys}"
        )

    if duplicate_keys > 0:
        raise ValueError(
            f"Duplicate primary keys found: {duplicate_keys}"
        )

    logger.info(
        "Validated %s records using primary key %s",
        total_records,
        primary_key
    )

    return total_records


# Validate patient records
patient_count = validate_dataset(
    patients_df,
    "patient_id"
)

assert patient_count == 5, "Unexpected patient count"

# Validate encounter records
encounter_count = validate_dataset(
    encounters_df,
    "encounter_id"
)

assert encounter_count == 7, "Unexpected encounter count"

# Validate patient ages
invalid_ages = patients_df.filter(
    (F.col("age") < 0) |
    (F.col("age") > 120) |
    F.col("age").isNull()
).count()

assert invalid_ages == 0, "Invalid patient ages found"

# Validate treatment costs
invalid_costs = encounters_df.filter(
    (F.col("treatment_cost") < 0) |
    F.col("treatment_cost").isNull() |
    F.isnan("treatment_cost")
).count()

assert invalid_costs == 0, "Invalid treatment costs found"

# Validate patient-to-encounter relationships
orphan_encounters = encounters_df.join(
    patients_df.select("patient_id"),
    on="patient_id",
    how="left_anti"
)

orphan_count = orphan_encounters.count()

assert orphan_count == 0, "Orphan encounters found"

print("PASS: Patient primary key validation")
print("PASS: Encounter primary key validation")
print("PASS: Patient age validation")
print("PASS: Treatment cost validation")
print("PASS: Referential integrity validation")
print("All healthcare data quality checks passed")


INFO:healthcare_analytics:Validated 5 records using primary key patient_id
INFO:healthcare_analytics:Validated 7 records using primary key encounter_id


PASS: Patient primary key validation
PASS: Encounter primary key validation
PASS: Patient age validation
PASS: Treatment cost validation
PASS: Referential integrity validation
All healthcare data quality checks passed


In [0]:
%python

# Define fully qualified Delta table names
catalog_name = "workspace"
schema_name = "healthcare_analytics"

patients_table = f"{catalog_name}.{schema_name}.patients"
encounters_table = f"{catalog_name}.{schema_name}.encounters"

# Save validated patient records as a managed Delta table
patients_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(patients_table)

print(f"Patient Delta table created: {patients_table}")

# Save validated encounter records as a managed Delta table
encounters_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(encounters_table)

print(f"Encounter Delta table created: {encounters_table}")

# Verify the persisted record counts
persisted_patients = spark.table(patients_table).count()
persisted_encounters = spark.table(encounters_table).count()

assert persisted_patients == 5, "Patient table count mismatch"
assert persisted_encounters == 7, "Encounter table count mismatch"

print("Persisted patient records:", persisted_patients)
print("Persisted encounter records:", persisted_encounters)
print("PASS: Both Delta tables saved and verified")


Patient Delta table created: workspace.healthcare_analytics.patients
Encounter Delta table created: workspace.healthcare_analytics.encounters
Persisted patient records: 5
Persisted encounter records: 7
PASS: Both Delta tables saved and verified


In [0]:
%python

# Inspect the Delta table storage formats
for table_name in [
    "workspace.healthcare_analytics.patients",
    "workspace.healthcare_analytics.encounters"
]:
    table_details = spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    )

    table_format = table_details.select("format").first()[0]

    assert table_format == "delta", (
        f"Unexpected table format for {table_name}: {table_format}"
    )

    print(f"PASS: {table_name} uses Delta format")


PASS: workspace.healthcare_analytics.patients uses Delta format
PASS: workspace.healthcare_analytics.encounters uses Delta format


In [0]:
%python

from pyspark.sql import functions as F

# Read source data from persisted Delta tables
patients_source_df = spark.table(
    "workspace.healthcare_analytics.patients"
)

encounters_source_df = spark.table(
    "workspace.healthcare_analytics.encounters"
)

# Aggregate healthcare encounters by patient
encounter_summary_df = (
    encounters_source_df
    .groupBy("patient_id")
    .agg(
        F.count("encounter_id").alias("total_encounters"),
        F.round(
            F.sum("treatment_cost"), 2
        ).alias("total_treatment_cost"),
        F.max("encounter_date").alias("last_encounter_date")
    )
)

# Join patient details with encounter metrics
patient_analytics_df = (
    patients_source_df
    .join(
        encounter_summary_df,
        on="patient_id",
        how="left"
    )
    .select(
        "patient_id",
        "patient_name",
        "age",
        "gender",
        "city",
        F.coalesce(
            F.col("total_encounters"),
            F.lit(0)
        ).alias("total_encounters"),
        F.coalesce(
            F.col("total_treatment_cost"),
            F.lit(0.0)
        ).alias("total_treatment_cost"),
        "last_encounter_date"
    )
)

# Display the analytics dataset
display(
    patient_analytics_df.orderBy("patient_id")
)

print(
    "Total analytics records:",
    patient_analytics_df.count()
)


patient_id,patient_name,age,gender,city,total_encounters,total_treatment_cost,last_encounter_date
P001,Patient Alpha,34,Female,Bengaluru,2,3700.0,2026-04-05
P002,Patient Beta,46,Male,Mysuru,2,7300.0,2026-04-09
P003,Patient Gamma,28,Female,Hyderabad,1,3600.0,2026-04-06
P004,Patient Delta,62,Male,Chennai,1,5400.0,2026-04-07
P005,Patient Epsilon,51,Female,Pune,1,1800.0,2026-04-08


Total analytics records: 5


In [0]:
%python

# Define the target analytics table
analytics_table = (
    "workspace.healthcare_analytics.patient_analytics"
)

# Persist analytics results as a managed Delta table
patient_analytics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(analytics_table)

# Read the persisted analytics table
saved_analytics_df = spark.table(analytics_table)

# Verify record counts and financial metrics
analytics_count = saved_analytics_df.count()

total_encounters = saved_analytics_df.agg(
    F.sum("total_encounters")
).first()[0]

total_cost = saved_analytics_df.agg(
    F.sum("total_treatment_cost")
).first()[0]

assert analytics_count == 5, "Analytics record count mismatch"
assert total_encounters == 7, "Encounter count mismatch"
assert round(total_cost, 2) == 21800.00, (
    "Treatment cost reconciliation failed"
)

# Verify that the target table uses Delta format
table_format = spark.sql(
    f"DESCRIBE DETAIL {analytics_table}"
).select("format").first()[0]

assert table_format == "delta", "Target table is not Delta"

print("PASS: Patient analytics Delta table created")
print("Analytics records:", analytics_count)
print("Total encounters:", total_encounters)
print("Total treatment cost:", round(total_cost, 2))
print("Storage format:", table_format)


PASS: Patient analytics Delta table created
Analytics records: 5
Total encounters: 7
Total treatment cost: 21800.0
Storage format: delta


In [0]:
%python

catalog_name = "workspace"
schema_name = "healthcare_analytics"
table_name = "patient_analytics"

print("Checking healthcare analytics schema...")

spark.sql("""
    SHOW TABLES IN workspace.healthcare_analytics
""").show(truncate=False)

table_exists = spark.catalog.tableExists(
    f"{catalog_name}.{schema_name}.{table_name}"
)

print("Patient analytics table exists:", table_exists)

if table_exists:
    print("Patient analytics record count:",
          spark.table(
              "workspace.healthcare_analytics.patient_analytics"
          ).count())
else:
    print("Patient analytics table has not been created.")


Checking healthcare analytics schema...
+--------------------+-------------------------------+-----------+
|database            |tableName                      |isTemporary|
+--------------------+-------------------------------+-----------+
|healthcare_analytics|bronze_patient_events          |false      |
|healthcare_analytics|catalog_validation             |false      |
|healthcare_analytics|encounters                     |false      |
|healthcare_analytics|patient_analytics              |false      |
|healthcare_analytics|patients                       |false      |
|healthcare_analytics|pipeline_execution_audit       |false      |
|healthcare_analytics|workflow_execution_log         |false      |
|healthcare_analytics|workflow_patient_metrics       |false      |
|healthcare_analytics|workflow_performance_diagnostic|false      |
+--------------------+-------------------------------+-----------+

Patient analytics table exists: True
Patient analytics record count: 5


In [0]:
%sql

SELECT
    city,
    COUNT(*) AS total_patients,
    SUM(total_encounters) AS total_encounters,
    ROUND(SUM(total_treatment_cost), 2) AS total_treatment_cost
FROM workspace.healthcare_analytics.patient_analytics
GROUP BY city
ORDER BY total_treatment_cost DESC;


city,total_patients,total_encounters,total_treatment_cost
Mysuru,1,2,7300.0
Chennai,1,1,5400.0
Bengaluru,1,2,3700.0
Hyderabad,1,1,3600.0
Pune,1,1,1800.0


Databricks visualization. Run in Databricks to view.

In [0]:
%sql

SELECT
    department,
    COUNT(*) AS total_encounters,
    COUNT(DISTINCT patient_id) AS unique_patients,
    ROUND(SUM(treatment_cost), 2) AS total_treatment_cost,
    ROUND(AVG(treatment_cost), 2) AS average_treatment_cost
FROM workspace.healthcare_analytics.encounters
GROUP BY department
ORDER BY total_treatment_cost DESC;


department,total_encounters,unique_patients,total_treatment_cost,average_treatment_cost
Cardiology,3,3,11000.0,3666.67
Neurology,1,1,4200.0,4200.0
Orthopedics,1,1,3600.0,3600.0
General Medicine,2,2,3000.0,1500.0


In [0]:
%sql

SELECT
    patient_id,
    patient_name,
    city,
    total_encounters,
    total_treatment_cost,
    last_encounter_date
FROM workspace.healthcare_analytics.patient_analytics
ORDER BY total_treatment_cost DESC;


patient_id,patient_name,city,total_encounters,total_treatment_cost,last_encounter_date
P002,Patient Beta,Mysuru,2,7300.0,2026-04-09
P004,Patient Delta,Chennai,1,5400.0,2026-04-07
P001,Patient Alpha,Bengaluru,2,3700.0,2026-04-05
P003,Patient Gamma,Hyderabad,1,3600.0,2026-04-06
P005,Patient Epsilon,Pune,1,1800.0,2026-04-08


In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM workspace.healthcare_analytics.patients)
        AS total_patients,

    (SELECT COUNT(*)
     FROM workspace.healthcare_analytics.encounters)
        AS total_encounters,

    (SELECT COUNT(*)
     FROM workspace.healthcare_analytics.patient_analytics)
        AS analytics_records,

    (SELECT ROUND(SUM(treatment_cost), 2)
     FROM workspace.healthcare_analytics.encounters)
        AS total_treatment_cost;


total_patients,total_encounters,analytics_records,total_treatment_cost
5,7,5,21800.0


In [0]:
%python

tables = [
    "workspace.healthcare_analytics.patients",
    "workspace.healthcare_analytics.encounters",
    "workspace.healthcare_analytics.patient_analytics"
]

for table_name in tables:
    print(f"\nDelta transaction history: {table_name}")

    history_df = spark.sql(
        f"DESCRIBE HISTORY {table_name}"
    )

    display(
        history_df.select(
            "version",
            "timestamp",
            "operation"
        ).orderBy("version", ascending=False)
    )



Delta transaction history: workspace.healthcare_analytics.patients


version,timestamp,operation
1,2026-11-17T19:44:59.000Z,CREATE OR REPLACE TABLE AS SELECT
0,2026-11-17T17:52:46.000Z,CREATE OR REPLACE TABLE AS SELECT



Delta transaction history: workspace.healthcare_analytics.encounters


version,timestamp,operation
1,2026-11-17T19:45:02.000Z,CREATE OR REPLACE TABLE AS SELECT
0,2026-11-17T17:52:50.000Z,CREATE OR REPLACE TABLE AS SELECT



Delta transaction history: workspace.healthcare_analytics.patient_analytics


version,timestamp,operation
2,2026-11-17T19:45:12.000Z,CREATE OR REPLACE TABLE AS SELECT
1,2026-11-17T17:56:25.000Z,CREATE OR REPLACE TABLE AS SELECT
0,2026-11-17T17:55:08.000Z,CREATE OR REPLACE TABLE AS SELECT


In [0]:
%python

for table_name in tables:
    details = spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).first().asDict()

    assert details["format"] == "delta", (
        f"Invalid storage format: {table_name}"
    )

    print(f"PASS: {table_name}")
    print("Format:", details["format"])
    print("Files:", details["numFiles"])
    print("Size in bytes:", details["sizeInBytes"])
    print("-" * 50)


PASS: workspace.healthcare_analytics.patients
Format: delta
Files: 1
Size in bytes: 2113
--------------------------------------------------
PASS: workspace.healthcare_analytics.encounters
Format: delta
Files: 1
Size in bytes: 1962
--------------------------------------------------
PASS: workspace.healthcare_analytics.patient_analytics
Format: delta
Files: 1
Size in bytes: 2701
--------------------------------------------------


In [0]:
%python

from pyspark.sql import functions as F

catalog = "workspace"
schema = "healthcare_analytics"

patients = spark.table(f"{catalog}.{schema}.patients")
encounters = spark.table(f"{catalog}.{schema}.encounters")
analytics = spark.table(f"{catalog}.{schema}.patient_analytics")

# Record count validation
assert patients.count() == 5
assert encounters.count() == 7
assert analytics.count() == 5

# Primary key uniqueness
assert patients.select("patient_id").distinct().count() == 5
assert encounters.select("encounter_id").distinct().count() == 7
assert analytics.select("patient_id").distinct().count() == 5

# Referential integrity
orphan_count = encounters.join(
    patients.select("patient_id"),
    on="patient_id",
    how="left_anti"
).count()

assert orphan_count == 0

# Financial reconciliation
source_cost = encounters.agg(
    F.sum("treatment_cost")
).first()[0]

analytics_cost = analytics.agg(
    F.sum("total_treatment_cost")
).first()[0]

assert round(source_cost, 2) == 21800.00
assert round(analytics_cost, 2) == 21800.00

# Encounter reconciliation
analytics_visits = analytics.agg(
    F.sum("total_encounters")
).first()[0]

assert analytics_visits == 7

# Verify all tables are Delta
for table_name in [
    f"{catalog}.{schema}.patients",
    f"{catalog}.{schema}.encounters",
    f"{catalog}.{schema}.patient_analytics"
]:
    storage_format = spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).select("format").first()[0]

    assert storage_format == "delta"

print("PASS: Patient record count")
print("PASS: Encounter record count")
print("PASS: Analytics record count")
print("PASS: Primary key uniqueness")
print("PASS: Referential integrity")
print("PASS: Financial reconciliation")
print("PASS: Encounter reconciliation")
print("PASS: Delta storage verification")
print("ALL HEALTHCARE ANALYTICS VALIDATIONS PASSED")


PASS: Patient record count
PASS: Encounter record count
PASS: Analytics record count
PASS: Primary key uniqueness
PASS: Referential integrity
PASS: Financial reconciliation
PASS: Encounter reconciliation
PASS: Delta storage verification
ALL HEALTHCARE ANALYTICS VALIDATIONS PASSED


In [0]:
%sql

SELECT
    table_catalog,
    table_schema,
    table_name,
    table_type
FROM workspace.information_schema.tables
WHERE table_schema = 'healthcare_analytics'
ORDER BY table_name;


table_catalog,table_schema,table_name,table_type
workspace,healthcare_analytics,bronze_patient_events,MANAGED
workspace,healthcare_analytics,catalog_validation,MANAGED
workspace,healthcare_analytics,encounters,MANAGED
workspace,healthcare_analytics,patient_analytics,MANAGED
workspace,healthcare_analytics,patients,MANAGED
workspace,healthcare_analytics,pipeline_execution_audit,MANAGED
workspace,healthcare_analytics,workflow_execution_log,MANAGED
workspace,healthcare_analytics,workflow_patient_metrics,MANAGED
workspace,healthcare_analytics,workflow_performance_diagnostic,MANAGED


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)
from delta.tables import DeltaTable
import json
import logging

# --------------------------------------------------
# Healthcare Delta Lake and Unity Catalog validation
# --------------------------------------------------

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("healthcare_catalog_validation")

CATALOG = "workspace"
SCHEMA = "healthcare_analytics"

PATIENTS = f"{CATALOG}.{SCHEMA}.patients"
ENCOUNTERS = f"{CATALOG}.{SCHEMA}.encounters"
ANALYTICS = f"{CATALOG}.{SCHEMA}.patient_analytics"
VALIDATION = f"{CATALOG}.{SCHEMA}.catalog_validation"

results = {}
failures = []

def record_check(name, passed, details=None):
    results[name] = {
        "status": "PASS" if passed else "FAIL",
        "details": details
    }

    if passed:
        print(f"PASS: {name}")
    else:
        failures.append(name)
        print(f"FAIL: {name}")

def run_optional_check(name, action):
    try:
        result = action()
        results[name] = {
            "status": "PASS",
            "details": result
        }
        print(f"PASS: {name}")
        return result
    except Exception as exc:
        results[name] = {
            "status": "UNAVAILABLE",
            "details": str(exc)[:500]
        }
        print(f"UNAVAILABLE: {name}")
        print(str(exc)[:300])
        return None

print("=" * 65)
print("HEALTHCARE DELTA LAKE AND UNITY CATALOG VALIDATION")
print("=" * 65)

# --------------------------------------------------
# 1. Verify catalog and schema
# --------------------------------------------------

print("\nCATALOG AND SCHEMA DISCOVERY")

catalogs = [
    row.catalog
    for row in spark.sql("SHOW CATALOGS").collect()
]

record_check(
    "Workspace catalog exists",
    CATALOG in catalogs,
    catalogs
)

assert CATALOG in catalogs, "Workspace catalog unavailable"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

active_catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

active_schema = spark.sql(
    "SELECT current_schema()"
).first()[0]

record_check(
    "Active catalog and schema",
    active_catalog == CATALOG and active_schema == SCHEMA,
    f"{active_catalog}.{active_schema}"
)

assert active_catalog == CATALOG
assert active_schema == SCHEMA

# --------------------------------------------------
# 2. Discover existing healthcare tables
# --------------------------------------------------

print("\nHEALTHCARE TABLE DISCOVERY")

required_tables = [
    PATIENTS,
    ENCOUNTERS,
    ANALYTICS
]

for table_name in required_tables:
    exists = spark.catalog.tableExists(table_name)

    record_check(
        f"Table exists: {table_name}",
        exists
    )

    assert exists, f"Missing table: {table_name}"

# --------------------------------------------------
# 3. Inspect Unity Catalog table metadata
# --------------------------------------------------

print("\nUNITY CATALOG METADATA")

def inspect_information_schema():
    metadata = spark.sql(f"""
        SELECT
            table_catalog,
            table_schema,
            table_name,
            table_type
        FROM `{CATALOG}`.information_schema.tables
        WHERE table_schema = '{SCHEMA}'
        ORDER BY table_name
    """)

    display(metadata)

    return {
        "visible_tables": metadata.count()
    }

run_optional_check(
    "Information schema inspection",
    inspect_information_schema
)

# --------------------------------------------------
# 4. Verify Delta Lake storage
# --------------------------------------------------

print("\nDELTA TABLE STORAGE VALIDATION")

for table_name in required_tables:
    details = spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).first().asDict()

    record_check(
        f"Delta format: {table_name}",
        details["format"] == "delta",
        {
            "format": details["format"],
            "num_files": details["numFiles"],
            "size_bytes": details["sizeInBytes"]
        }
    )

    assert details["format"] == "delta"

# --------------------------------------------------
# 5. Validate existing healthcare data
# --------------------------------------------------

print("\nHEALTHCARE DATA QUALITY")

patients_df = spark.table(PATIENTS)
encounters_df = spark.table(ENCOUNTERS)
analytics_df = spark.table(ANALYTICS)

patient_count = patients_df.count()
encounter_count = encounters_df.count()
analytics_count = analytics_df.count()

record_check(
    "Patient record count",
    patient_count == 5,
    patient_count
)

record_check(
    "Encounter record count",
    encounter_count == 7,
    encounter_count
)

record_check(
    "Analytics record count",
    analytics_count == 5,
    analytics_count
)

assert patient_count == 5
assert encounter_count == 7
assert analytics_count == 5

patient_duplicates = (
    patients_df.groupBy("patient_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

encounter_duplicates = (
    encounters_df.groupBy("encounter_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

record_check(
    "Patient primary key uniqueness",
    patient_duplicates == 0
)

record_check(
    "Encounter primary key uniqueness",
    encounter_duplicates == 0
)

assert patient_duplicates == 0
assert encounter_duplicates == 0

orphan_count = encounters_df.join(
    patients_df.select("patient_id"),
    on="patient_id",
    how="left_anti"
).count()

record_check(
    "Encounter referential integrity",
    orphan_count == 0,
    orphan_count
)

assert orphan_count == 0

source_cost = encounters_df.agg(
    F.sum("treatment_cost").alias("total")
).first()["total"]

analytics_cost = analytics_df.agg(
    F.sum("total_treatment_cost").alias("total")
).first()["total"]

record_check(
    "Healthcare financial reconciliation",
    round(source_cost, 2) == 21800.00
    and round(analytics_cost, 2) == 21800.00,
    {
        "source_cost": source_cost,
        "analytics_cost": analytics_cost
    }
)

assert round(source_cost, 2) == 21800.00
assert round(analytics_cost, 2) == 21800.00

# --------------------------------------------------
# 6. Inspect existing Delta transaction history
# --------------------------------------------------

print("\nDELTA TRANSACTION HISTORY")

for table_name in required_tables:
    history = spark.sql(
        f"DESCRIBE HISTORY {table_name}"
    )

    history_count = history.count()

    record_check(
        f"Transaction history: {table_name}",
        history_count >= 1,
        {"versions_found": history_count}
    )

    assert history_count >= 1

    display(
        history.select(
            "version",
            "timestamp",
            "operation"
        ).orderBy(F.desc("version")).limit(5)
    )

# --------------------------------------------------
# 7. Create isolated synthetic Delta validation table
# --------------------------------------------------

print("\nISOLATED DELTA VALIDATION")

validation_schema = StructType([
    StructField("record_id", StringType(), False),
    StructField("validation_status", StringType(), False),
    StructField("version_number", IntegerType(), False)
])

initial_data = [
    ("VAL001", "PENDING", 1),
    ("VAL002", "PENDING", 1)
]

initial_df = spark.createDataFrame(
    initial_data,
    validation_schema
)

# Only the dedicated validation table is overwritten.
# Existing healthcare source tables remain untouched.

initial_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(VALIDATION)

baseline_version = spark.sql(
    f"DESCRIBE HISTORY {VALIDATION}"
).agg(
    F.max("version").alias("latest_version")
).first()["latest_version"]

record_check(
    "Validation Delta table created",
    spark.table(VALIDATION).count() == 2,
    {"baseline_version": baseline_version}
)

assert spark.table(VALIDATION).count() == 2

# --------------------------------------------------
# 8. Execute Delta MERGE
# --------------------------------------------------

print("\nDELTA MERGE OPERATION")

updates_df = spark.createDataFrame(
    [
        ("VAL001", "APPROVED", 2),
        ("VAL003", "APPROVED", 2)
    ],
    validation_schema
)

delta_target = DeltaTable.forName(
    spark,
    VALIDATION
)

(
    delta_target.alias("target")
    .merge(
        updates_df.alias("source"),
        "target.record_id = source.record_id"
    )
    .whenMatchedUpdate(set={
        "validation_status": "source.validation_status",
        "version_number": "source.version_number"
    })
    .whenNotMatchedInsert(values={
        "record_id": "source.record_id",
        "validation_status": "source.validation_status",
        "version_number": "source.version_number"
    })
    .execute()
)

merged_df = spark.table(VALIDATION)

display(
    merged_df.orderBy("record_id")
)

merged_count = merged_df.count()

approved_count = merged_df.filter(
    F.col("validation_status") == "APPROVED"
).count()

record_check(
    "Delta MERGE execution",
    merged_count == 3 and approved_count == 2,
    {
        "total_records": merged_count,
        "approved_records": approved_count
    }
)

assert merged_count == 3
assert approved_count == 2

# --------------------------------------------------
# 9. Verify Delta Time Travel
# --------------------------------------------------

print("\nDELTA TIME TRAVEL VALIDATION")

historical_df = spark.sql(
    f"""
    SELECT *
    FROM {VALIDATION}
    VERSION AS OF {baseline_version}
    """
)

historical_count = historical_df.count()

historical_pending = historical_df.filter(
    F.col("validation_status") == "PENDING"
).count()

record_check(
    "Delta Time Travel",
    historical_count == 2 and historical_pending == 2,
    {
        "historical_version": baseline_version,
        "historical_records": historical_count
    }
)

assert historical_count == 2
assert historical_pending == 2

print("Historical Delta snapshot:")
display(historical_df.orderBy("record_id"))

# --------------------------------------------------
# 10. Inspect catalog permissions
# --------------------------------------------------

print("\nUNITY CATALOG ACCESS INSPECTION")

def inspect_grants():
    grants_df = spark.sql(
        f"SHOW GRANTS ON SCHEMA `{CATALOG}`.`{SCHEMA}`"
    )

    display(grants_df)

    return {
        "visible_grants": grants_df.count()
    }

run_optional_check(
    "Unity Catalog schema grants",
    inspect_grants
)

# --------------------------------------------------
# 11. Verify final validation table state
# --------------------------------------------------

print("\nFINAL DELTA VALIDATION")

final_records = {
    row["record_id"]: (
        row["validation_status"],
        row["version_number"]
    )
    for row in spark.table(VALIDATION).collect()
}

expected_records = {
    "VAL001": ("APPROVED", 2),
    "VAL002": ("PENDING", 1),
    "VAL003": ("APPROVED", 2)
}

record_check(
    "Final Delta record integrity",
    final_records == expected_records,
    final_records
)

assert final_records == expected_records

latest_version = spark.sql(
    f"DESCRIBE HISTORY {VALIDATION}"
).agg(
    F.max("version").alias("latest_version")
).first()["latest_version"]

record_check(
    "Delta transaction version advanced",
    latest_version > baseline_version,
    {
        "baseline_version": baseline_version,
        "latest_version": latest_version
    }
)

assert latest_version > baseline_version

# --------------------------------------------------
# 12. Final execution report
# --------------------------------------------------

print("\n" + "=" * 65)
print("HEALTHCARE DATA PLATFORM VALIDATION REPORT")
print("=" * 65)

print(json.dumps(results, indent=2, default=str))

passed = sum(
    item["status"] == "PASS"
    for item in results.values()
)

unavailable = sum(
    item["status"] == "UNAVAILABLE"
    for item in results.values()
)

failed = sum(
    item["status"] == "FAIL"
    for item in results.values()
)

print("\nSUMMARY")
print("Passed checks:", passed)
print("Unavailable optional checks:", unavailable)
print("Failed checks:", failed)

if failures:
    raise AssertionError(
        "Healthcare catalog validation failed: "
        + ", ".join(failures)
    )

print("\nSUCCESS: All required Delta and catalog checks passed")
print("Existing healthcare source tables were not modified")
print("Validation table:", VALIDATION)


HEALTHCARE DELTA LAKE AND UNITY CATALOG VALIDATION

CATALOG AND SCHEMA DISCOVERY
PASS: Workspace catalog exists
PASS: Active catalog and schema

HEALTHCARE TABLE DISCOVERY
PASS: Table exists: workspace.healthcare_analytics.patients
PASS: Table exists: workspace.healthcare_analytics.encounters
PASS: Table exists: workspace.healthcare_analytics.patient_analytics

UNITY CATALOG METADATA


table_catalog,table_schema,table_name,table_type
workspace,healthcare_analytics,bronze_patient_events,MANAGED
workspace,healthcare_analytics,catalog_validation,MANAGED
workspace,healthcare_analytics,encounters,MANAGED
workspace,healthcare_analytics,patient_analytics,MANAGED
workspace,healthcare_analytics,patients,MANAGED
workspace,healthcare_analytics,pipeline_execution_audit,MANAGED
workspace,healthcare_analytics,workflow_execution_log,MANAGED
workspace,healthcare_analytics,workflow_patient_metrics,MANAGED
workspace,healthcare_analytics,workflow_performance_diagnostic,MANAGED


PASS: Information schema inspection

DELTA TABLE STORAGE VALIDATION
PASS: Delta format: workspace.healthcare_analytics.patients
PASS: Delta format: workspace.healthcare_analytics.encounters
PASS: Delta format: workspace.healthcare_analytics.patient_analytics

HEALTHCARE DATA QUALITY
PASS: Patient record count
PASS: Encounter record count
PASS: Analytics record count
PASS: Patient primary key uniqueness
PASS: Encounter primary key uniqueness
PASS: Encounter referential integrity
PASS: Healthcare financial reconciliation

DELTA TRANSACTION HISTORY
PASS: Transaction history: workspace.healthcare_analytics.patients


version,timestamp,operation
1,2026-11-17T19:44:59.000Z,CREATE OR REPLACE TABLE AS SELECT
0,2026-11-17T17:52:46.000Z,CREATE OR REPLACE TABLE AS SELECT


PASS: Transaction history: workspace.healthcare_analytics.encounters


version,timestamp,operation
1,2026-11-17T19:45:02.000Z,CREATE OR REPLACE TABLE AS SELECT
0,2026-11-17T17:52:50.000Z,CREATE OR REPLACE TABLE AS SELECT


... (5 outputs left out)

record_id,validation_status,version_number
VAL001,PENDING,1
VAL002,PENDING,1



UNITY CATALOG ACCESS INSPECTION


Principal,ActionType,ObjectType,ObjectKey


PASS: Unity Catalog schema grants

FINAL DELTA VALIDATION
PASS: Final Delta record integrity
PASS: Delta transaction version advanced

HEALTHCARE DATA PLATFORM VALIDATION REPORT
{
  "Workspace catalog exists": {
    "status": "PASS",
    "details": [
      "samples",
      "system",
      "workspace"
    ]
  },
  "Active catalog and schema": {
    "status": "PASS",
    "details": "workspace.healthcare_analytics"
  },
  "Table exists: workspace.healthcare_analytics.patients": {
    "status": "PASS",
    "details": null
  },
  "Table exists: workspace.healthcare_analytics.encounters": {
    "status": "PASS",
    "details": null
  },
  "Table exists: workspace.healthcare_analytics.patient_analytics": {
    "status": "PASS",
    "details": null
  },
  "Information schema inspection": {
    "status": "PASS",
    "details": {
      "visible_tables": 9
    }
  },
  "Delta format: workspace.healthcare_analytics.patients": {
    "status": "PASS",
    "details": {
      "format": "delta",
      

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)
from pathlib import Path
import json

# =========================================================
# HEALTHCARE INCREMENTAL INGESTION WITH DATABRICKS AUTO LOADER
# =========================================================

CATALOG = "workspace"
SCHEMA = "healthcare_analytics"
VOLUME = "healthcare_ingestion"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_patient_events"

VOLUME_NAME = f"{CATALOG}.{SCHEMA}.{VOLUME}"

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

SOURCE_PATH = f"{BASE_PATH}/landing/patient_events"
SCHEMA_PATH = f"{BASE_PATH}/metadata/patient_events_schema"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/patient_events_bronze"

print("=" * 65)
print("HEALTHCARE AUTO LOADER PIPELINE")
print("=" * 65)

# ---------------------------------------------------------
# 1. Verify Unity Catalog namespace
# ---------------------------------------------------------

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

assert spark.catalog.tableExists(
    f"{CATALOG}.{SCHEMA}.patients"
)

print("PASS: Healthcare Unity Catalog namespace verified")

# ---------------------------------------------------------
# 2. Create a managed Unity Catalog volume
# ---------------------------------------------------------

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS
`{CATALOG}`.`{SCHEMA}`.`{VOLUME}`
""")

print("PASS: Managed ingestion volume available")

# ---------------------------------------------------------
# 3. Define synthetic healthcare event schema
# ---------------------------------------------------------

event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("patient_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("department", StringType(), True),
    StructField("treatment_cost", DoubleType(), True),
    StructField("event_version", IntegerType(), True)
])

# ---------------------------------------------------------
# 4. Create source directories
# ---------------------------------------------------------

Path(SOURCE_PATH).mkdir(
    parents=True,
    exist_ok=True
)

print("PASS: Landing directory available")

# ---------------------------------------------------------
# 5. Create initial synthetic events
# ---------------------------------------------------------

initial_events = [
    {
        "event_id": "PE001",
        "patient_id": "P001",
        "event_type": "CONSULTATION",
        "department": "Cardiology",
        "treatment_cost": 2500.0,
        "event_version": 1
    },
    {
        "event_id": "PE002",
        "patient_id": "P002",
        "event_type": "DIAGNOSTIC",
        "department": "Neurology",
        "treatment_cost": 4200.0,
        "event_version": 1
    },
    {
        "event_id": "PE003",
        "patient_id": "P003",
        "event_type": "CONSULTATION",
        "department": "Orthopedics",
        "treatment_cost": 3600.0,
        "event_version": 1
    }
]

def write_json_events(filename, events):
    file_path = Path(SOURCE_PATH) / filename

    # Preserve existing landing files on reruns.
    if file_path.exists():
        print(f"Existing source file retained: {filename}")
        return

    with file_path.open("x", encoding="utf-8") as file:
        for event in events:
            file.write(json.dumps(event) + "\n")

    print(f"Created source file: {filename}")

write_json_events(
    "patient_events_initial.json",
    initial_events
)

print("PASS: Initial synthetic events prepared")

# ---------------------------------------------------------
# 6. Define Auto Loader ingestion
# ---------------------------------------------------------

def run_auto_loader():

    streaming_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option(
            "cloudFiles.schemaLocation",
            SCHEMA_PATH
        )
        .schema(event_schema)
        .load(SOURCE_PATH)
    )

    bronze_df = (
        streaming_df
        .withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "source_file",
            F.col("_metadata.file_path")
        )
    )

    query = (
        bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH
        )
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
    )

    query.awaitTermination()

    if query.exception() is not None:
        raise RuntimeError(str(query.exception()))

# ---------------------------------------------------------
# 7. Process initial events
# ---------------------------------------------------------

print("\nINITIAL AUTO LOADER INGESTION")

run_auto_loader()

bronze_df = spark.table(BRONZE_TABLE)

initial_ids = ["PE001", "PE002", "PE003"]

initial_count = (
    bronze_df
    .filter(F.col("event_id").isin(initial_ids))
    .count()
)

assert initial_count == 3, (
    f"Expected 3 initial records, found {initial_count}"
)

print("PASS: Initial ingestion completed")

# ---------------------------------------------------------
# 8. Add a new incremental file
# ---------------------------------------------------------

print("\nINCREMENTAL FILE ARRIVAL")

incremental_events = [
    {
        "event_id": "PE004",
        "patient_id": "P004",
        "event_type": "CONSULTATION",
        "department": "Cardiology",
        "treatment_cost": 5400.0,
        "event_version": 1
    },
    {
        "event_id": "PE005",
        "patient_id": "P005",
        "event_type": "FOLLOW_UP",
        "department": "General Medicine",
        "treatment_cost": 1800.0,
        "event_version": 1
    }
]

write_json_events(
    "patient_events_incremental.json",
    incremental_events
)

# ---------------------------------------------------------
# 9. Resume using the same checkpoint
# ---------------------------------------------------------

print("\nRESUMING AUTO LOADER")

run_auto_loader()

expected_ids = [
    "PE001", "PE002", "PE003",
    "PE004", "PE005"
]

result_df = spark.table(BRONZE_TABLE).filter(
    F.col("event_id").isin(expected_ids)
)

display(
    result_df.select(
        "event_id",
        "patient_id",
        "event_type",
        "department",
        "treatment_cost",
        "source_file",
        "ingestion_timestamp"
    ).orderBy("event_id")
)

assert result_df.count() == 5

print("PASS: Incremental ingestion completed")

# ---------------------------------------------------------
# 10. Verify checkpoint recovery
# ---------------------------------------------------------

print("\nCHECKPOINT RECOVERY TEST")

before_restart = result_df.count()

run_auto_loader()

after_restart = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_id").isin(expected_ids))
    .count()
)

assert before_restart == after_restart == 5

print("PASS: Restart did not reprocess existing files")

# ---------------------------------------------------------
# 11. Validate duplicate prevention
# ---------------------------------------------------------

duplicates = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_id").isin(expected_ids))
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicates == 0

print("PASS: No duplicate synthetic event IDs")

# ---------------------------------------------------------
# 12. Validate data quality
# ---------------------------------------------------------

quality_df = spark.table(BRONZE_TABLE).filter(
    F.col("event_id").isin(expected_ids)
)

null_ids = quality_df.filter(
    F.col("event_id").isNull()
    | F.col("patient_id").isNull()
).count()

negative_costs = quality_df.filter(
    F.col("treatment_cost") < 0
).count()

assert null_ids == 0
assert negative_costs == 0

print("PASS: Data quality validation")

# ---------------------------------------------------------
# 13. Validate patient references
# ---------------------------------------------------------

known_patients = spark.table(
    f"{CATALOG}.{SCHEMA}.patients"
).select("patient_id")

orphan_events = (
    quality_df
    .join(
        known_patients,
        on="patient_id",
        how="left_anti"
    )
    .count()
)

assert orphan_events == 0

print("PASS: Patient referential integrity")

# ---------------------------------------------------------
# 14. Inspect Delta transaction history
# ---------------------------------------------------------

print("\nBRONZE DELTA TRANSACTION HISTORY")

history_df = spark.sql(
    f"DESCRIBE HISTORY {BRONZE_TABLE}"
)

display(
    history_df.select(
        "version",
        "timestamp",
        "operation"
    ).orderBy(F.desc("version"))
)

assert history_df.count() >= 1

print("PASS: Delta transaction history available")

# ---------------------------------------------------------
# 15. Verify storage metadata
# ---------------------------------------------------------

details = spark.sql(
    f"DESCRIBE DETAIL {BRONZE_TABLE}"
).first().asDict()

assert details["format"] == "delta"

print("PASS: Bronze table uses Delta Lake")

# ---------------------------------------------------------
# 16. Final pipeline summary
# ---------------------------------------------------------

print("\n" + "=" * 65)
print("HEALTHCARE INCREMENTAL INGESTION SUMMARY")
print("=" * 65)

print("Catalog:", CATALOG)
print("Schema:", SCHEMA)
print("Managed volume:", VOLUME_NAME)
print("Bronze table:", BRONZE_TABLE)
print("Expected synthetic events:", 5)
print("Verified synthetic events:", quality_df.count())
print("Duplicate event IDs:", duplicates)
print("Orphan patient references:", orphan_events)
print("Null key records:", null_ids)
print("Negative treatment costs:", negative_costs)

print("\nSUCCESS: AUTO LOADER INGESTION VALIDATED")
print("SUCCESS: INCREMENTAL FILE PROCESSING VALIDATED")
print("SUCCESS: CHECKPOINT RESTART VALIDATED")
print("SUCCESS: BRONZE DELTA TABLE VALIDATED")
print("SUCCESS: EXISTING HEALTHCARE TABLES PRESERVED")


HEALTHCARE AUTO LOADER PIPELINE
PASS: Healthcare Unity Catalog namespace verified
PASS: Managed ingestion volume available
PASS: Landing directory available
Existing source file retained: patient_events_initial.json
PASS: Initial synthetic events prepared

INITIAL AUTO LOADER INGESTION
PASS: Initial ingestion completed

INCREMENTAL FILE ARRIVAL
Existing source file retained: patient_events_incremental.json

RESUMING AUTO LOADER


event_id,patient_id,event_type,department,treatment_cost,source_file,ingestion_timestamp
PE001,P001,CONSULTATION,Cardiology,2500.0,/Volumes/workspace/healthcare_analytics/healthcare_ingestion/landing/patient_events/patient_events_initial.json,2026-11-17T18:31:54.069Z
PE002,P002,DIAGNOSTIC,Neurology,4200.0,/Volumes/workspace/healthcare_analytics/healthcare_ingestion/landing/patient_events/patient_events_initial.json,2026-11-17T18:31:54.069Z
PE003,P003,CONSULTATION,Orthopedics,3600.0,/Volumes/workspace/healthcare_analytics/healthcare_ingestion/landing/patient_events/patient_events_initial.json,2026-11-17T18:31:54.069Z
PE004,P004,CONSULTATION,Cardiology,5400.0,/Volumes/workspace/healthcare_analytics/healthcare_ingestion/landing/patient_events/patient_events_incremental.json,2026-11-17T18:32:03.378Z
PE005,P005,FOLLOW_UP,General Medicine,1800.0,/Volumes/workspace/healthcare_analytics/healthcare_ingestion/landing/patient_events/patient_events_incremental.json,2026-11-17T18:32:03.378Z


PASS: Incremental ingestion completed

CHECKPOINT RECOVERY TEST
PASS: Restart did not reprocess existing files
PASS: No duplicate synthetic event IDs
PASS: Data quality validation
PASS: Patient referential integrity

BRONZE DELTA TRANSACTION HISTORY


version,timestamp,operation
2,2026-11-17T18:32:06.000Z,STREAMING UPDATE
1,2026-11-17T18:31:58.000Z,STREAMING UPDATE
0,2026-11-17T18:31:49.000Z,CREATE TABLE


PASS: Delta transaction history available
PASS: Bronze table uses Delta Lake

HEALTHCARE INCREMENTAL INGESTION SUMMARY
Catalog: workspace
Schema: healthcare_analytics
Managed volume: workspace.healthcare_analytics.healthcare_ingestion
Bronze table: workspace.healthcare_analytics.bronze_patient_events
Expected synthetic events: 5
Verified synthetic events: 5
Duplicate event IDs: 0
Orphan patient references: 0
Null key records: 0
Negative treatment costs: 0

SUCCESS: AUTO LOADER INGESTION VALIDATED
SUCCESS: INCREMENTAL FILE PROCESSING VALIDATED
SUCCESS: CHECKPOINT RESTART VALIDATED
SUCCESS: BRONZE DELTA TABLE VALIDATED
SUCCESS: EXISTING HEALTHCARE TABLES PRESERVED


In [0]:

from datetime import datetime, timezone
from time import perf_counter, sleep
from uuid import uuid4
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)

# ==================================================
# HEALTHCARE WORKFLOW ORCHESTRATION AND OPTIMIZATION
# ==================================================

CATALOG = "workspace"
SCHEMA = "healthcare_analytics"

PATIENTS = f"{CATALOG}.{SCHEMA}.patients"
ENCOUNTERS = f"{CATALOG}.{SCHEMA}.encounters"
BRONZE = f"{CATALOG}.{SCHEMA}.bronze_patient_events"

WORKFLOW_RESULTS = (
    f"{CATALOG}.{SCHEMA}.workflow_patient_metrics"
)

EXECUTION_LOG = (
    f"{CATALOG}.{SCHEMA}.workflow_execution_log"
)

run_id = str(uuid4())
execution_records = []
task_results = {}
task_attempts = {}

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def execute_task(name, action, dependencies=(), max_attempts=2):
    for dependency in dependencies:
        if dependency not in task_results:
            raise RuntimeError(
                f"{name} requires successful task {dependency}"
            )

    for attempt in range(1, max_attempts + 1):
        started_at = utc_now()
        start = perf_counter()

        try:
            result = action()
            duration = round(perf_counter() - start, 4)

            execution_records.append((
                run_id, name, attempt, "SUCCESS",
                started_at, utc_now(), duration, ""
            ))

            task_results[name] = result
            task_attempts[name] = attempt

            print(
                f"PASS: {name} "
                f"(attempt {attempt}, {duration}s)"
            )
            return result

        except Exception as exc:
            duration = round(perf_counter() - start, 4)

            execution_records.append((
                run_id, name, attempt, "FAILED",
                started_at, utc_now(), duration,
                str(exc)[:1000]
            ))

            print(
                f"FAILED: {name}, attempt {attempt}: {exc}"
            )

            if attempt == max_attempts:
                raise

            sleep(1)

# --------------------------------------------------
# TASK 1: Validate source tables
# --------------------------------------------------

def validate_sources():
    required = [PATIENTS, ENCOUNTERS, BRONZE]

    for table in required:
        assert spark.catalog.tableExists(table), (
            f"Missing source table: {table}"
        )

    patients = spark.table(PATIENTS)
    encounters = spark.table(ENCOUNTERS)
    bronze = spark.table(BRONZE)

    assert patients.count() == 5
    assert encounters.count() == 7

    expected_events = [
        "PE001", "PE002", "PE003",
        "PE004", "PE005"
    ]

    events = bronze.filter(
        F.col("event_id").isin(expected_events)
    )

    assert events.count() == 5
    assert events.select("event_id").distinct().count() == 5

    return {
        "patients": 5,
        "encounters": 7,
        "bronze_events": 5
    }

# --------------------------------------------------
# TASK 2: Data quality and reconciliation
# --------------------------------------------------

def validate_quality():
    patients = spark.table(PATIENTS)
    encounters = spark.table(ENCOUNTERS)

    assert (
        patients.select("patient_id").distinct().count()
        == patients.count()
    )

    assert (
        encounters.select("encounter_id").distinct().count()
        == encounters.count()
    )

    orphans = encounters.join(
        patients.select("patient_id"),
        "patient_id",
        "left_anti"
    ).count()

    assert orphans == 0

    total_cost = encounters.agg(
        F.sum("treatment_cost")
    ).first()[0]

    assert round(total_cost, 2) == 21800.00

    return {
        "orphan_records": orphans,
        "total_treatment_cost": float(total_cost)
    }

# --------------------------------------------------
# TASK 3: Optimize a patient-encounter join
# --------------------------------------------------

def build_patient_metrics():
    patients = spark.table(PATIENTS)
    encounters = spark.table(ENCOUNTERS)

    # The patient dimension is small in this synthetic
    # dataset, so broadcasting avoids a large-side shuffle.

    joined = encounters.join(
        F.broadcast(
            patients.select("patient_id", "city")
        ),
        on="patient_id",
        how="inner"
    )

    metrics = (
        joined.groupBy("patient_id", "city")
        .agg(
            F.count("encounter_id").alias(
                "total_encounters"
            ),
            F.round(
                F.sum("treatment_cost"), 2
            ).alias("total_treatment_cost")
        )
    )

    # Materialize results to validate the aggregation.
    rows = metrics.collect()

    assert len(rows) == 5
    assert sum(r["total_encounters"] for r in rows) == 7

    total = sum(
        float(r["total_treatment_cost"])
        for r in rows
    )

    assert round(total, 2) == 21800.00

    return spark.createDataFrame(rows, metrics.schema)

# --------------------------------------------------
# TASK 4: Publish validated Delta output
# --------------------------------------------------

def publish_metrics():
    metrics = task_results["build_patient_metrics"]

    # Overwrite only this dedicated workflow output.
    # Source and previously created tables are preserved.
    (
        metrics.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(WORKFLOW_RESULTS)
    )

    saved = spark.table(WORKFLOW_RESULTS)

    assert saved.count() == 5

    return {
        "table": WORKFLOW_RESULTS,
        "records": 5
    }

# --------------------------------------------------
# TASK 5: Verify Delta output
# --------------------------------------------------

def verify_output():
    details = spark.sql(
        f"DESCRIBE DETAIL {WORKFLOW_RESULTS}"
    ).first().asDict()

    assert details["format"] == "delta"

    result = spark.table(WORKFLOW_RESULTS)

    assert result.count() == 5

    return {
        "format": details["format"],
        "records": result.count()
    }

# --------------------------------------------------
# EXECUTE DEPENDENCY-AWARE WORKFLOW
# --------------------------------------------------

print("=" * 65)
print("HEALTHCARE WORKFLOW EXECUTION")
print("=" * 65)
print("Run ID:", run_id)

workflow_error = None

try:
    execute_task(
        "validate_sources",
        validate_sources
    )

    execute_task(
        "validate_quality",
        validate_quality,
        dependencies=("validate_sources",)
    )

    execute_task(
        "build_patient_metrics",
        build_patient_metrics,
        dependencies=("validate_quality",)
    )

    execute_task(
        "publish_metrics",
        publish_metrics,
        dependencies=("build_patient_metrics",)
    )

    execute_task(
        "verify_output",
        verify_output,
        dependencies=("publish_metrics",)
    )

except Exception as exc:
    workflow_error = exc

# --------------------------------------------------
# PERSIST EXECUTION LOG
# --------------------------------------------------

log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("task_name", StringType(), False),
    StructField("attempt", IntegerType(), False),
    StructField("status", StringType(), False),
    StructField("started_at", StringType(), False),
    StructField("finished_at", StringType(), False),
    StructField("duration_seconds", DoubleType(), False),
    StructField("error_message", StringType(), False)
])

if execution_records:
    log_df = spark.createDataFrame(
        execution_records, log_schema
    )

    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(EXECUTION_LOG)
    )

    print("\nWORKFLOW EXECUTION LOG")

    display(
        spark.table(EXECUTION_LOG)
        .filter(F.col("run_id") == run_id)
        .orderBy("started_at", "task_name")
    )

if workflow_error is not None:
    raise workflow_error

# --------------------------------------------------
# VERIFY DEPENDENCIES AND EXECUTION HISTORY
# --------------------------------------------------

expected_order = [
    "validate_sources",
    "validate_quality",
    "build_patient_metrics",
    "publish_metrics",
    "verify_output"
]

actual_order = [
    record[1]
    for record in execution_records
    if record[3] == "SUCCESS"
]

assert actual_order == expected_order

assert all(
    record[3] == "SUCCESS"
    for record in execution_records
)

assert len(task_results) == 5

print("\nWORKFLOW OUTPUT")
display(
    spark.table(WORKFLOW_RESULTS)
    .orderBy("patient_id")
)

print("\n" + "=" * 65)
print("WORKFLOW VALIDATION SUMMARY")
print("=" * 65)

print("Run ID:", run_id)
print("Successful tasks:", len(task_results))
print("Failed attempts:", sum(
    r[3] == "FAILED" for r in execution_records
))
print("Execution log:", EXECUTION_LOG)
print("Analytics output:", WORKFLOW_RESULTS)

print("\nPASS: Task dependencies validated")
print("PASS: Execution order validated")
print("PASS: Bounded retry logic configured")
print("PASS: Execution history persisted")
print("PASS: Broadcast join used")
print("PASS: Delta analytics output verified")

print("\nSUCCESS: HEALTHCARE WORKFLOW VALIDATED")


HEALTHCARE WORKFLOW EXECUTION
Run ID: 5c2f8460-1260-4a97-a0e7-0a9fa8ec25d8
PASS: validate_sources (attempt 1, 2.2466s)
PASS: validate_quality (attempt 1, 2.4587s)
PASS: build_patient_metrics (attempt 1, 0.9629s)
PASS: publish_metrics (attempt 1, 2.351s)
PASS: verify_output (attempt 1, 1.067s)

WORKFLOW EXECUTION LOG


run_id,task_name,attempt,status,started_at,finished_at,duration_seconds,error_message
5c2f8460-1260-4a97-a0e7-0a9fa8ec25d8,validate_sources,1,SUCCESS,2026-11-17T19:46:47.247217+00:00,2026-11-17T19:46:49.493851+00:00,2.2466,
5c2f8460-1260-4a97-a0e7-0a9fa8ec25d8,validate_quality,1,SUCCESS,2026-11-17T19:46:49.493931+00:00,2026-11-17T19:46:51.952684+00:00,2.4587,
5c2f8460-1260-4a97-a0e7-0a9fa8ec25d8,build_patient_metrics,1,SUCCESS,2026-11-17T19:46:51.953097+00:00,2026-11-17T19:46:52.915973+00:00,0.9629,
5c2f8460-1260-4a97-a0e7-0a9fa8ec25d8,publish_metrics,1,SUCCESS,2026-11-17T19:46:52.916106+00:00,2026-11-17T19:46:55.267073+00:00,2.351,
5c2f8460-1260-4a97-a0e7-0a9fa8ec25d8,verify_output,1,SUCCESS,2026-11-17T19:46:55.267291+00:00,2026-11-17T19:46:56.334336+00:00,1.067,



WORKFLOW OUTPUT


patient_id,city,total_encounters,total_treatment_cost
P001,Bengaluru,2,3700.0
P002,Mysuru,2,7300.0
P003,Hyderabad,1,3600.0
P004,Chennai,1,5400.0
P005,Pune,1,1800.0



WORKFLOW VALIDATION SUMMARY
Run ID: 5c2f8460-1260-4a97-a0e7-0a9fa8ec25d8
Successful tasks: 5
Failed attempts: 0
Execution log: workspace.healthcare_analytics.workflow_execution_log
Analytics output: workspace.healthcare_analytics.workflow_patient_metrics

PASS: Task dependencies validated
PASS: Execution order validated
PASS: Bounded retry logic configured
PASS: Execution history persisted
PASS: Broadcast join used
PASS: Delta analytics output verified

SUCCESS: HEALTHCARE WORKFLOW VALIDATED


In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone
from time import perf_counter
from uuid import uuid4
import json

# ====================================================
# HEALTHCARE DATA PLATFORM - MANAGED JOB NOTEBOOK
# ====================================================

CATALOG = "workspace"
SCHEMA = "healthcare_analytics"

PATIENTS = f"{CATALOG}.{SCHEMA}.patients"
ENCOUNTERS = f"{CATALOG}.{SCHEMA}.encounters"
BRONZE = f"{CATALOG}.{SCHEMA}.bronze_patient_events"

METRICS = f"{CATALOG}.{SCHEMA}.workflow_patient_metrics"
AUDIT = f"{CATALOG}.{SCHEMA}.pipeline_execution_audit"

run_id = str(uuid4())
started_at = datetime.now(timezone.utc)
start_time = perf_counter()

status = "SUCCESS"
error_message = None
metrics_count = 0
bronze_count = 0

try:
    # ---------------------------------------------
    # Validate required source tables
    # ---------------------------------------------

    for table in [PATIENTS, ENCOUNTERS, BRONZE]:
        assert spark.catalog.tableExists(table), (
            f"Required table missing: {table}"
        )

    patients = spark.table(PATIENTS)
    encounters = spark.table(ENCOUNTERS)
    bronze = spark.table(BRONZE)

    assert patients.count() == 5
    assert encounters.count() == 7

    # ---------------------------------------------
    # Validate Bronze ingestion
    # ---------------------------------------------

    expected_events = [
        "PE001", "PE002", "PE003",
        "PE004", "PE005"
    ]

    bronze_events = bronze.filter(
        F.col("event_id").isin(expected_events)
    )

    bronze_count = bronze_events.count()

    assert bronze_count == 5

    assert (
        bronze_events.select("event_id")
        .distinct()
        .count() == 5
    )

    # ---------------------------------------------
    # Validate patient referential integrity
    # ---------------------------------------------

    orphans = encounters.join(
        patients.select("patient_id"),
        on="patient_id",
        how="left_anti"
    ).count()

    assert orphans == 0

    # ---------------------------------------------
    # Optimize the patient-encounter join
    # ---------------------------------------------

    joined = encounters.join(
        F.broadcast(
            patients.select("patient_id", "city")
        ),
        on="patient_id",
        how="inner"
    )

    metrics = (
        joined.groupBy("patient_id", "city")
        .agg(
            F.count("encounter_id").alias(
                "total_encounters"
            ),
            F.round(
                F.sum("treatment_cost"), 2
            ).alias("total_treatment_cost")
        )
    )

    metrics_count = metrics.count()

    assert metrics_count == 5

    total_cost = metrics.agg(
        F.sum("total_treatment_cost")
    ).first()[0]

    assert round(total_cost, 2) == 21800.00

    # ---------------------------------------------
    # Publish Delta analytics table
    # ---------------------------------------------

    (
        metrics.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(METRICS)
    )

    assert spark.table(METRICS).count() == 5

    print("PASS: Source validation")
    print("PASS: Bronze ingestion validation")
    print("PASS: Referential integrity")
    print("PASS: Broadcast join aggregation")
    print("PASS: Delta analytics publication")

except Exception as exc:
    status = "FAILED"
    error_message = str(exc)
    print("FAILED:", error_message)

finally:
    finished_at = datetime.now(timezone.utc)
    duration = round(perf_counter() - start_time, 3)

    audit_record = [{
        "run_id": run_id,
        "status": status,
        "started_at": started_at.isoformat(),
        "finished_at": finished_at.isoformat(),
        "duration_seconds": duration,
        "bronze_records": bronze_count,
        "metrics_records": metrics_count,
        "error_message": error_message
    }]

    audit_df = spark.createDataFrame(
        [
            (
                row["run_id"],
                row["status"],
                row["started_at"],
                row["finished_at"],
                row["duration_seconds"],
                row["bronze_records"],
                row["metrics_records"],
                row["error_message"]
            )
            for row in audit_record
        ],
        """
        run_id STRING,
        status STRING,
        started_at STRING,
        finished_at STRING,
        duration_seconds DOUBLE,
        bronze_records LONG,
        metrics_records LONG,
        error_message STRING
        """
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(AUDIT)
    )

    print(json.dumps(audit_record, indent=2))

if status != "SUCCESS":
    raise RuntimeError(
        f"Healthcare pipeline failed: {error_message}"
    )

print("\nSUCCESS: HEALTHCARE MANAGED JOB NOTEBOOK VALIDATED")


PASS: Source validation
PASS: Bronze ingestion validation
PASS: Referential integrity
PASS: Broadcast join aggregation
PASS: Delta analytics publication
[
  {
    "run_id": "9ad779c7-a686-43e9-9d85-5377889562cf",
    "status": "SUCCESS",
    "started_at": "2026-11-17T19:47:00.188891+00:00",
    "finished_at": "2026-11-17T19:47:07.541522+00:00",
    "duration_seconds": 7.353,
    "bronze_records": 5,
    "metrics_records": 5,
    "error_message": null
  }
]

SUCCESS: HEALTHCARE MANAGED JOB NOTEBOOK VALIDATED
